# 01 – Train forecast model

Predict `production_next_quarter` from `gold.ml_crop_forecast_features`.
Split is chronological. The winner is the lowest validation MAE.
The fitted winner is saved with joblib to `MODEL_PATH`.

In [0]:
import inspect
import os

import joblib
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from configs.project_config import (
    GOLD_TABLES,
    MODEL_PATH,
    TRAIN_END_YEAR,
    VALIDATION_YEARS,
    TEST_YEAR,
    NUMERIC_FEATURES,
    BOOLEAN_FEATURES,
    CATEGORICAL_FEATURES,
    LABEL_COL
)

try:
    from lightgbm import LGBMRegressor
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"])
    from lightgbm import LGBMRegressor

In [0]:
feature_cols = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES

pdf = (
    spark.table(GOLD_TABLES["ml_crop_forecast_features"])
    .where(f"{LABEL_COL} IS NOT NULL")
    .toPandas()
)
for col in BOOLEAN_FEATURES:
    pdf[col] = pdf[col].astype("float")

train = pdf[pdf["year"] <= TRAIN_END_YEAR]
valid = pdf[pdf["year"].isin(VALIDATION_YEARS)]
test = pdf[pdf["year"] == TEST_YEAR]
print(f"train {len(train)}  valid {len(valid)}  test {len(test)}")
if train.empty or valid.empty:
    raise ValueError("Train or validation split is empty. Check year coverage.")

In [0]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    nonzero = y_true != 0
    mape = (
        float(np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])))
        if nonzero.any()
        else None
    )
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": mape,
        "r2": float(r2_score(y_true, y_pred)),
    }

def preprocessor():
    encoder_kwargs = {"handle_unknown": "ignore"}
    params = inspect.signature(OneHotEncoder).parameters
    if "sparse_output" in params:
        encoder_kwargs["sparse_output"] = False
    else:
        encoder_kwargs["sparse"] = False
    encoder = OneHotEncoder(**encoder_kwargs)
    numeric = Pipeline([("impute", SimpleImputer(strategy="median"))])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder),
    ])
    return ColumnTransformer([
        ("num", numeric, NUMERIC_FEATURES + BOOLEAN_FEATURES),
        ("cat", categorical, CATEGORICAL_FEATURES),
    ])

class Lag4Baseline(BaseEstimator, RegressorMixin):
    """Next quarter equals the same quarter last year."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        values = X["production_lag_4"].astype(float)
        return values.fillna(0.0).to_numpy()

def candidate_models():
    return {
        "baseline_lag4": Lag4Baseline(),
        "linear_regression": Pipeline([
            ("prep", preprocessor()),
            ("model", LinearRegression()),
        ]),
        "random_forest": Pipeline([
            ("prep", preprocessor()),
            ("model", RandomForestRegressor(
                n_estimators=100,
                max_depth=12,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=42,
            )),
        ]),
        "lightgbm": Pipeline([
            ("prep", preprocessor()),
            ("model", LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                num_leaves=31,
                random_state=42,
                verbosity=-1,
            )),
        ]),
    }

In [0]:
X_train, y_train = train[feature_cols], train[LABEL_COL]
X_valid, y_valid = valid[feature_cols], valid[LABEL_COL]
X_test, y_test = test[feature_cols], test[LABEL_COL]

fitted = {}
results = []
for name, model in candidate_models().items():
    model.fit(X_train, y_train)
    valid_metrics = regression_metrics(y_valid, model.predict(X_valid))
    row = {"name": name, "valid_mae": valid_metrics["mae"]}
    if not test.empty:
        test_metrics = regression_metrics(y_test, model.predict(X_test))
        row["test_mae"] = test_metrics["mae"]
    fitted[name] = model
    results.append(row)
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in valid_metrics.items()})

winner = min(results, key=lambda row: row["valid_mae"])
winner_model = fitted[winner["name"]]
print("Winner:", winner)

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump({"model_name": winner["name"], "model": winner_model, "features": feature_cols}, MODEL_PATH)
print(f"Saved {MODEL_PATH}")
print("Next: 02_evaluate_model")